# 02 — Plasma OES Substrate Classification

**Mesbah Lab CAP** — SVM + Random Forest substrate classification on 51-channel N₂ OES data.

This notebook demonstrates:
1. Loading the Mesbah CAP dataset (substrate_type: glass=0, metal=1)
2. Training an SVM classifier with 5-fold cross-validation
3. Plotting the confusion matrix
4. Comparing SVM vs Random Forest

In [1]:
import os
import sys
import warnings
from pathlib import Path

nb_dir = Path.cwd()
if nb_dir.name == "notebooks":
    os.chdir(nb_dir.parent)
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

print("Working directory:", Path.cwd())

Working directory: C:\Users\PC\libs-spectral-analysis


## 1 · Load Mesbah CAP data

In [ ]:
from src.data_loader import load_mesbah_cap
from sklearn.preprocessing import StandardScaler

# Load Mesbah CAP dataset (substrate_type: glass=0, metal=1)
X, y, wl = load_mesbah_cap("data/mesbah_cap/dat_train.csv", target="substrate_type")
print(f"Dataset : {X.shape[0]} spectra × {X.shape[1]} channels")
print(f"Classes : glass={int((y==0).sum())}, metal={int((y==1).sum())}")

X_sc = StandardScaler().fit_transform(X)
print(f"Scaled  : {X_sc.shape}")

## 2 · SVM classification (5-fold cross-validation)

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=False, random_state=42)
y_pred_svm = cross_val_predict(svm, X_sc, y, cv=cv)

micro_f1_svm = f1_score(y, y_pred_svm, average="micro")
macro_f1_svm = f1_score(y, y_pred_svm, average="macro")
print(f"SVM  micro-F1 : {micro_f1_svm:.4f}")
print(f"SVM  macro-F1 : {macro_f1_svm:.4f}")

In [4]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=False, random_state=42)
y_pred_svm = cross_val_predict(svm, X_pca, y, cv=cv)

micro_f1_svm = f1_score(y, y_pred_svm, average="micro")
macro_f1_svm = f1_score(y, y_pred_svm, average="macro")
print(f"SVM  micro-F1 : {micro_f1_svm:.4f}")
print(f"SVM  macro-F1 : {macro_f1_svm:.4f}")

SVM  micro-F1 : 0.6198
SVM  macro-F1 : 0.5914


## 3 · Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm_mat = confusion_matrix(y, y_pred_svm)
class_names = ["Glass", "Metal"]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_mat, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(2))
ax.set_yticks(range(2))
ax.set_xticklabels(class_names, fontsize=10)
ax.set_yticklabels(class_names, fontsize=10)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title(f"SVM Confusion Matrix  (micro-F1 = {micro_f1_svm:.3f})")
for i in range(2):
    for j in range(2):
        ax.text(
            j, i, str(cm_mat[i, j]),
            ha="center", va="center", fontsize=14,
            color="white" if cm_mat[i, j] > cm_mat.max() * 0.5 else "black",
        )
plt.tight_layout()

out_path = Path("outputs/notebook_02_confusion_matrix.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(str(out_path), dpi=100, bbox_inches="tight")
plt.close(fig)
print(f"Saved → {out_path}")

## 4 · Random Forest comparison

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
y_pred_rf = cross_val_predict(rf, X_sc, y, cv=cv)

micro_f1_rf = f1_score(y, y_pred_rf, average="micro")
macro_f1_rf = f1_score(y, y_pred_rf, average="macro")
print(f"RF   micro-F1 : {micro_f1_rf:.4f}")
print(f"RF   macro-F1 : {macro_f1_rf:.4f}")

## Summary

| Model | micro-F1 (5-fold CV) |
|-------|----------------------|
| SVM (RBF, C=10) | See above |
| Random Forest (100 trees) | See above |

- Binary classification of substrate type (glass vs metal) using 51-channel N₂ OES features.
- Both SVM and RF operate on StandardScaler-normalised raw spectral channels.